# Recreating Harrison's IFS calibration data

### Initialization

In [45]:
import sys,os
import numpy as np
from crispy.tools.initLogger import getLogger
import glob
from crispy.tools.wavecal import buildcalibrations
# log = getLogger('crispy')

# Set our directory to the crispy install location
os.chdir("C:/Users/ebray/Github Repos/crispy/crispy/")
from crispy.configs.WFIRST.params import Params

# Initialize our Params object and print locations of a few directories
par = Params()
par.wavecalDir = r"C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox"
# par.exportDir = r"C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660\Evans_output_sandbox"
print(f'wavecalDir is {par.wavecalDir}')
print(f'exportDir is {par.exportDir}')
print(f'unitTestOutputs is {par.unitTestsOutputs}')

wavecalDir is C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox
exportDir is ../SimResults
unitTestOutputs is ../unitTestsOutputs


### Define some setup-specific parameters

In [ ]:
# Number of lenslets across array (account for rotation)
par.nlens = 108
par.pitch = 174e-6         # Lenslet pitch (meters)
par.interlace = 2.          # Interlacing
# Rotation angle of the lenslets (radians)
par.philens = np.arcsin(1. / np.sqrt(par.interlace**2 + 1))
par.lenslet_sampling = 1. / 2.  # lenslet size in lambda/D
par.lenslet_wav = 660.     # Wavelength at which lenslet sampling is defined (nm)
par.FWHMlam = 660.         # Lam at which FWHM is defined
par.pinhole = False       # Use a pinhole grid?
par.npix = 1024            # Number of pixels in final detector
par.pixsize = 13e-6        # Pixel size (meters)
par.R = 70
par.BW = 0.18

In [47]:
import re
# Match both "ds600.0nm.fit" and "det_600.fits" formats
# calibration_filelist = glob.glob(par.wavecalDir + '/ds*.fit*') + glob.glob(par.wavecalDir + '/det_*.fits')
calibration_filelist = glob.glob(par.wavecalDir + '/det_*.fits')
par.filelist = sorted(calibration_filelist)

# Extract wavelength from either format: "ds630.0nm.fit" -> 630.0 or "det_630.fits" -> 630
def extract_wavelength(filename):
    # Try format: ds<wavelength>nm.fit (e.g., "ds630.0nm.fit")
    m = re.search(r'ds([\d.]+)nm', filename)
    if m:
        return float(m.group(1))
    # Try format: det_<wavelength>.fits (e.g., "det_630.fits")
    m = re.search(r'det_(\d{3})\.fits', filename)
    if m:
        return float(m.group(1))
    return None

calibration_wavelengths = [int(extract_wavelength(filename)) for filename in calibration_filelist]
print('calibration_filelist ->', par.filelist)
print('calibration_wavelengths ->', calibration_wavelengths)

calibration_filelist -> ['C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_600.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_610.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_620.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_630.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_640.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_650.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660_sandbox\\det_660.fits', 'C:\\Users\\ebray\\Box\\ExoSpec-shared\\1_IFS\\PISCES\\Cal_Data\\CRISPY\\Calibration\\wavecalR70_660

In [48]:
buildcalibrations(par,
                lamlist=calibration_wavelengths, # list of wavelengths at which the calibration files were taken
                inspect=True,         # if True, constructs a bunch of images to verify a good calibration
                genwavelengthsol=True,  # Compute wavelength at the center of all pixels
                makehiresPSFlets=True,  # this requires very high SNR on the monochromatic frames
                makePSFWidths=True,
                makePolychrome=True,   # This is needed to use least squares extraction
                upsample=3,            # upsampling factor of the high-resolution PSFLets
                nsubarr=3,             # the detector is divided into nsubarr^2 regions for PSFLet averaging
                apodize=False,          # to match PSFlet spot locations, only use the inner circular part of the
                                        # detector, hence discarding the corners of the detector where lenslets are
                                        # distorted
                threshold=1e-4,
                parallel=True,
                  )

crispy - INFO - Building calibration files; placing results in C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox
crispy - INFO - Read data from HDU 1 of C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox\det_600.fits
crispy - INFO - Read data from HDU 1 of C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox\det_600.fits
crispy - INFO - Read data from HDU 1 of C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox\det_600.fits
crispy - INFO - Read data from HDU 1 of C:\Users\ebray\Box\ExoSpec-shared\1_IFS\PISCES\Cal_Data\CRISPY\Calibration\wavecalR70_660_sandbox\det_600.fits
crispy - INFO - Mean, median, std: (0.00033170995, 1.6616417e-11, 0.0014635939)
crispy - INFO - Mean, median, std: (0.00033170995, 1.6616417e-11, 0.0014635939)
crispy - INFO - Initializing PSFlet location transformation coefficients
cr

In [49]:
plt.close('all')